# Model Training

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import pickle

import warnings
warnings.filterwarnings('ignore')

In [2]:
from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PowerTransformer, StandardScaler

from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import VotingRegressor, RandomForestRegressor,GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import SGDRegressor

In [3]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import joblib
from sklearn.pipeline import Pipeline

In [4]:
import time
from sklearn.model_selection import cross_val_score

In [5]:
vendor_invoice = pd.read_csv('vendor_invoice.csv')

### Train-Test Split

In [6]:
vendor_invoice.columns

Index(['VendorNumber', 'VendorName', 'InvoiceDate', 'PONumber', 'PODate',
       'PayDate', 'Quantity', 'Dollars', 'Freight', 'Approval'],
      dtype='object')

In [7]:
X = vendor_invoice[['Quantity', 'Dollars']]
y = vendor_invoice['Freight']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Load Feature Engineering Pipeline

In [8]:
with open('feature_engineering_pipeline.pkl', 'rb') as f:
    preprocessing_loaded = pickle.load(f)

In [9]:
preprocessing_loaded

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num_trf',
                                                  Pipeline(steps=[('scaling',
                                                                   StandardScaler())]),
                                                  ['Quantity', 'Dollars'])]))])

### Transform Features

In [10]:
X_train_trf = preprocessing_loaded.fit_transform(X_train)
X_test_trf = preprocessing_loaded.transform(X_test)

In [11]:
joblib.dump(preprocessing_loaded,'fitted_pipeline.pkl')

['fitted_pipeline.pkl']

### Model Training & Evaluation

In [12]:
models = {
    'LinearRegression': LinearRegression(),
    'DecisionTreeRegressor': DecisionTreeRegressor(),
    'RandomForestRegressor': RandomForestRegressor(),
    'XGBRegressor': XGBRegressor()
}

results = {}

n = X_test_trf.shape[0]     
p = X_test_trf.shape[1]     

for name, model in models.items():
    start_time = time.time()
    model.fit(X_train_trf, y_train)
    end_time = time.time()
    y_pred = model.predict(X_test_trf)
    
    training_time = end_time - start_time
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

    results[name] = {
            "MSE": mse,
            "RMSE": rmse,
            "MAE": mae,
            "R2": r2,
            "Adj R2": adj_r2,
            "Training Time": training_time
        }

results_df = pd.DataFrame(results).T
print("Model Performance Metrics:\n")
print(results_df)

best_model = results_df["Adj R2"].idxmax()
print(f"\nBest model based on R²: {best_model}")

Model Performance Metrics:

                                MSE        RMSE        MAE        R2  \
LinearRegression       15482.522465  124.428785  24.459181  0.970020   
DecisionTreeRegressor  32270.982028  179.641259  33.968872  0.937511   
RandomForestRegressor  19676.120507  140.271596  28.254835  0.961899   
XGBRegressor           25756.170532  160.487291  28.011047  0.950126   

                         Adj R2  Training Time  
LinearRegression       0.969966       0.016315  
DecisionTreeRegressor  0.937398       0.080761  
RandomForestRegressor  0.961830       5.156831  
XGBRegressor           0.950036       0.345785  

Best model based on R²: LinearRegression


### Cross Validation

In [30]:
scores = cross_val_score(LinearRegression(), X_train, y_train, cv=5, scoring='r2')
scores.mean()

0.9695346955129442

In [31]:
scores

array([0.97450995, 0.97203677, 0.9247388 , 0.9931364 , 0.98325156])

### Hyperparameter Tuning of XGBoost

In [15]:
params = {
    'n_estimators': [100, 300, 500],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.7, 1.0],
    'colsample_bytree': [0.7, 1.0]
}

grid_lr = GridSearchCV(estimator=XGBRegressor(),param_grid=params,cv=5,scoring='r2',n_jobs=-1)

grid_lr.fit(X_train_trf, y_train)

GridSearchCV(cv=5,
             estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=False, eval_metric=None,
                                    feature_types=None, feature_weights=None,
                                    gamma=None, grow_policy=None,
                                    importance_type=None,
                                    interaction_constraints=None...
                                    max_cat_to_onehot=None, max_delta_step=None,
                                    max_depth=None, max_leaves=None,
                                    min_child_weight=None, missing=nan,
                                    monotone_constraints=None,
                                    multi_strategy=None, n_estimators=None,
                                    n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.7, 1.0],
                         'learning_rate': [0.01, 0.1, 0.2],
                         'max_depth': [3, 5, 7],
                         'n_estimators': [100, 300, 500],
                         'subsample': [0.7, 1.0]},
             scoring='r2')

In [16]:
grid_lr.best_params_, grid_lr.best_score_

({'colsample_bytree': 1.0,
  'learning_rate': 0.01,
  'max_depth': 3,
  'n_estimators': 500,
  'subsample': 1.0},
 0.9609986987388532)

In [17]:
params = {
    'n_estimators': [100, 300, 500],
    'max_depth': [3, 5, 7]
}

grid_rf = GridSearchCV(estimator=RandomForestRegressor(),param_grid=params,cv=5,scoring='r2',n_jobs=-1)

grid_rf.fit(X_train_trf, y_train)  

GridSearchCV(cv=5, estimator=RandomForestRegressor(), n_jobs=-1,
             param_grid={'max_depth': [3, 5, 7],
                         'n_estimators': [100, 300, 500]},
             scoring='r2')

In [18]:
grid_rf.best_params_, grid_rf.best_score_

({'max_depth': 3, 'n_estimators': 300}, 0.9626019284449617)

R2 score improved from 0.950126 to 0.9609986987388532 by XGBoost hyperparameter tuning. <br>
R2 score improved from 0.962096 to 0.9627084848838061 by Random forest hyperparameter tuning.

Since Freight is highly correlated with Dollars and Quantity, the relationship is nearly linear. That’s why Linear Regression outperformed more complex models like RandomForest and XGBoost even after tuning.

### Save the model

In [36]:
final_model = LinearRegression()
final_model.fit(X_train_trf, y_train)

LinearRegression()

In [37]:
joblib.dump(final_model,'final_model.pkl')

['final_model.pkl']

### Prediction

In [38]:
final_model_loaded = joblib.load('final_model.pkl')

In [39]:
y = final_model_loaded.predict(X_train_trf)
r2_score(y_train,y)

0.9707792745727539

In [40]:
y = final_model_loaded.predict(X_test_trf)
r2_score(y, y_test)

0.9692629627537986

### ML Flow

#### Model Tracking

In [13]:
import mlflow

In [16]:
#mlflow.set_experiment('Exp1')
#mlflow.set_tracking_uri('http://127.0.0.1:5000/')

#with mlflow.start_run():
    ##mlflow.log_params(params)
    #mlflow.log_metrics({"MSE": 15482.522465,
     #   "RMSE": 124.428785,
      #  "MAE": 24.459181,
      #  "R2": 0.970020,
     #   "Adj R2": 0.969966})
    #mlflow.sklearn.log_model(final_model,'Linear Regression')

In [ ]:
mlflow.set_experiment('Freight Cost Prediction v2')
mlflow.set_tracking_uri('http://127.0.0.1:5000/')

params = {
    'LinearRegression':{},
    'DecisionTreeRegressor':{'max_depth': 5},
    'RandomForestRegressor' :{'n_estimators': 100, 'max_depth': 10},
    'XGBRegressor':{'n_estimators': 100, 'learning_rate': 0.1}
}

models = {
    'LinearRegression': LinearRegression(**params['LinearRegression']),
    'DecisionTreeRegressor': DecisionTreeRegressor(**params['DecisionTreeRegressor']),
    'RandomForestRegressor': RandomForestRegressor(**params['RandomForestRegressor']),
    'XGBRegressor': XGBRegressor(**params['XGBRegressor'])
}

results = {}

n = X_test_trf.shape[0]     
p = X_test_trf.shape[1]     

for name, model in models.items():
    with mlflow.start_run(run_name=name):

        start_time = time.time()
        model.fit(X_train_trf, y_train)
        end_time = time.time()
        y_pred = model.predict(X_test_trf)
        
        training_time = end_time - start_time
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

        mlflow.log_param('model_name', name)
        for param_name, param_value in params[name].items():
            mlflow.log_param(param_name, param_value)
        #mlflow.log_params(model.get_params()): it logs all the model parameters, even the unused ones

        mlflow.log_metric("MSE", mse)
        mlflow.log_metric("RMSE", rmse)
        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("R2", r2)
        mlflow.log_metric("Adj_R2", adj_r2)
        mlflow.log_metric("Training_Time", training_time)

        mlflow.sklearn.log_model(model, name)

        results[name] = {
                "MSE": mse,
                "RMSE": rmse,
                "MAE": mae,
                "R2": r2,
                "Adj R2": adj_r2,
                "Training Time": training_time
            }

results_df = pd.DataFrame(results).T
print("Model Performance Metrics:\n")
print(results_df)

best_model = results_df["Adj R2"].idxmax()
print(f"\nBest model based on Adj R²: {best_model}")

2026/03/29 08:46:09 INFO mlflow.tracking.fluent: Experiment with name 'Freight Cost Prediction v2' does not exist. Creating a new experiment.


2026/03/29 08:46:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/29 08:46:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LinearRegression at: http://127.0.0.1:5000/#/experiments/3/runs/b91b617f2f0b4ffc9f6cd121011650e4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/03/29 08:46:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/29 08:46:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run DecisionTreeRegressor at: http://127.0.0.1:5000/#/experiments/3/runs/9614eefaf5e44d6fada95d97f6efb622
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/03/29 08:46:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/29 08:46:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RandomForestRegressor at: http://127.0.0.1:5000/#/experiments/3/runs/e88c5f30adef4b62942232d16485d0cd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/03/29 08:46:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/29 08:46:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBRegressor at: http://127.0.0.1:5000/#/experiments/3/runs/8f33b69b94c6428cbb4fcd2f7a995f1b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Model Performance Metrics:

                                MSE        RMSE        MAE        R2  \
LinearRegression       15482.522465  124.428785  24.459181  0.970020   
DecisionTreeRegressor  22680.199312  150.599467  33.367266  0.956082   
RandomForestRegressor  19535.464416  139.769326  27.710527  0.962172   
XGBRegressor           24188.780307  155.527426  26.960895  0.953161   

                         Adj R2  Training Time  
LinearRegression       0.969966       0.015715  
DecisionTreeRegressor  0.956003       0.031172  
RandomForestRegressor  0.962103       2.955976  
XGBRegressor           0.953076       0.183472  

Best model based on R²: LinearRegression


In [ ]:
# to restore deleted experiment
#from mlflow.tracking import MlflowClient

#client = MlflowClient()

#exp = client.get_experiment_by_name("Exp1")

#if exp:
   # client.restore_experiment(exp.experiment_id)

#### Model Registry

In [ ]:
registered_model_name = 'final_prediction_model'
run_id = 'b91b617f2f0b4ffc9f6cd121011650e4'
final_model_name = 'LinearRegression'

model_uri = f'runs:/{run_id}/{final_model_name}' # Trying to use the best model's run ID to register it

mlflow.register_model(
    model_uri=model_uri,
    name=registered_model_name
)

Successfully registered model 'final_prediction_model'.
2026/03/29 08:55:29 WARNING mlflow.tracking._model_registry.fluent: Run with id b91b617f2f0b4ffc9f6cd121011650e4 has no artifacts at artifact path 'LinearRegression', registering model based on models:/m-2cb26841e644479b978c2142f625a132 instead
2026/03/29 08:55:29 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: final_prediction_model, version 1
Created version '1' of model 'final_prediction_model'.


<ModelVersion: aliases=[], creation_timestamp=1774754729443, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1774754729443, metrics=None, model_id=None, name='final_prediction_model', params=None, run_id='b91b617f2f0b4ffc9f6cd121011650e4', run_link='', source='models:/m-2cb26841e644479b978c2142f625a132', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [22]:
# delete registered model
#from mlflow.tracking import MlflowClient

#client = MlflowClient()

#client.delete_registered_model("Freight_Price_Linear_Regression")

#### Load Model From MLFlow

In [20]:
registered_model_name = 'final_prediction_model'
model_version = 1
model_uri = f'models:/{registered_model_name}/{model_version}' # Trying to get the registered model 

loaded_model = mlflow.sklearn.load_model(model_uri)
y_pred = loaded_model.predict(X_test_trf)

print(y_pred[:4])
print(r2_score(y_test, y_pred))

[   7.3493271  1113.01346644    8.77140038   10.54528626]
0.9700197330175596


#### DEV to PROD

In [21]:
registered_model_name = 'final_prediction_model'
model_version = 1

dev_model_uri = f'models:/{registered_model_name}/{model_version}'
prod_model = 'final_prediction_model-prod'

client = mlflow.MlflowClient()

client.copy_model_version(src_model_uri=dev_model_uri, dst_name=prod_model)

Successfully registered model 'final_prediction_model-prod'.
Copied version '1' of model 'final_prediction_model' to version '1' of model 'final_prediction_model-prod'.


<ModelVersion: aliases=[], creation_timestamp=1774754924635, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1774754924635, metrics=None, model_id=None, name='final_prediction_model-prod', params=None, run_id='b91b617f2f0b4ffc9f6cd121011650e4', run_link='', source='models:/final_prediction_model/1', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

#### Load PROD Model From MLFlow

In [22]:
prod_model = 'final_prediction_model-prod'
model_version = 1

model_uri = f'models:/{prod_model}/{model_version}'

loaded_model = mlflow.sklearn.load_model(model_uri)
y_pred = loaded_model.predict(X_test_trf)

print(y_pred[:4])
print(r2_score(y_test, y_pred))

[   7.3493271  1113.01346644    8.77140038   10.54528626]
0.9700197330175596
